In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from all_imports import *
from gpu_warm_up import gpu_warmup
from text_cleaner import TextCleaner
from compute_metrics import compute_metrics
from multimodal_architecture_updated import *
import datasets as hf_datasets
#from extract_image_features import *

In [ ]:
logging.set_verbosity_error()

In [ ]:
def set_all_seeds(seed_value):
    """
    Sets all relevant random seeds to ensure reproducibility across runs.
    Covers Python's random, NumPy, PyTorch (CPU and CUDA), cuDNN, and
    the HuggingFace Transformers set_seed utility.

    Args:
        seed_value: Integer seed to apply globally.
    """
    random.seed(seed_value)
    np.random.seed(seed_value)
    if torch.cuda.is_available():
        torch.manual_seed(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    else:
        torch.manual_seed(seed_value)
    set_seed(seed_value)

In [ ]:
# Uncomment the desired model_name to switch text encoders.

# model_name = "ibm-granite/granite-embedding-30m-english"
# model_name = 'FacebookAI/roberta-base'
# model_name = 'GroNLP/hateBERT'
# model_name = 'google-bert/bert-base-uncased'
# model_name = 'answerdotai/ModernBERT-base'
# model_name = 'sentence-transformers/all-MiniLM-L6-v2'
model_name = 'distilbert-base-uncased'

tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# Maps model identifiers to (display_name, pooling_strategy, fusion_layer_sizes).
# Models with hidden_size < 512 start fusion layers at 256.

MODEL_CONFIGS = {
    'google-bert/bert-base-uncased':              ('BaseBERT',       'mean', [512, 256, 128, 64]),
    'GroNLP/hateBERT':                            ('HateBERT',       'mean', [512, 256, 128, 64]),
    'FacebookAI/roberta-base':                    ('RoBERTa',        'mean', [512, 256, 128, 64]),
    'answerdotai/ModernBERT-base':                ('ModernBERT',     'mean', [512, 256, 128, 64]),
    'distilbert-base-uncased':                    ('DistilBERT',     'mean', [512, 256, 128, 64]),
    'ibm-granite/granite-embedding-30m-english':  ('IBM Granite',    'mean', [256, 128, 64]),
    'sentence-transformers/all-MiniLM-L6-v2':     ('All-MiniLM-L6', 'mean', [256, 128, 64]),
}


In [ ]:
# TF32 gives a ~2x speedup over FP32 on Ampere GPUs with negligible accuracy loss.
torch.backends.cudnn.fp32_precision = 'tf32'
torch.set_float32_matmul_precision('high')
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
SEED = 41
set_all_seeds(SEED)
display_name, *_ = MODEL_CONFIGS[model_name]
safe_model_name = display_name.replace(' ', '_')  # Filesystem-safe name for output

In [ ]:
# Paths
ALL_DATA = "../Multi3Hate_Dataset/en.csv"
LABELS = "../Multi3Hate_Dataset/final_annotations.csv"
IMG_DIR   = "../Multi3Hate_Dataset/en/"

# Load datasets
df_data = pd.read_csv(ALL_DATA)
df_label = pd.read_csv(LABELS)

In [ ]:
# Selects only the relevant columns from each CSV (drops translation column
# since we work on English only), merges on Meme ID, and normalizes column names.
df_data  = df_data.iloc[:, [0, 2]]   # Keep meme_id and English text only
df_label = df_label.iloc[:, 0:2]     # Keep meme_id and label only

df_final = pd.merge(df_data, df_label, on='Meme ID', how='inner')
df_final = df_final.rename(columns={
    df_final.columns[0]: 'meme_id',
    df_final.columns[1]: 'sentence',
    df_final.columns[2]: 'label'
})
display(df_final.head())

In [ ]:
# If features haven't been precomputed and saved, run the following line to compute and save them.

#features = preprocess_and_save_features(IMG_DIR, df_data['Meme ID'], "multi3hate_features.npz")
#print(features)

In [ ]:
# Applies the project's custom TextCleaner to normalize meme text
# (e.g., lowercasing, punctuation handling, URL removal).

cleaner = TextCleaner()
df_final['sentence'] = df_final['sentence'].apply(cleaner.clean_text)

In [ ]:
# Computes balanced class weights from the full dataset and converts them to
# a tensor for use in the weighted CrossEntropyLoss during training.

from sklearn.utils.class_weight import compute_class_weight

train_labels = df_final['label'].tolist()

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(train_labels),
    y=train_labels
)

train_class_weights = torch.tensor(class_weights, dtype=torch.float32)

In [ ]:
def get_token_lengths(texts, tokenizer):
    """
    Computes the number of tokens per text sample (including special tokens,
    without truncation) to inform the max_length hyperparameter.

    Args:
        texts:     List of raw text strings.
        tokenizer: HuggingFace tokenizer to use for encoding.

    Returns:
        NumPy array of token counts, one per input text.
    """
    lengths = []
    for t in texts:
        enc = tokenizer(t, truncation=False, padding=False, add_special_tokens=True)
        lengths.append(len(enc["input_ids"]))
    return np.array(lengths)

lengths = get_token_lengths(df_final["sentence"].tolist(), tokenizer)

In [ ]:
# Visualizes the token length distribution with percentile markers
# to guide max_length selection.

import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.figure(figsize=(10, 6))

sns.histplot(lengths, bins=50, kde=True, color="skyblue", edgecolor="black", alpha=0.7)

percentile_90  = np.percentile(lengths, 90)
percentile_95  = np.percentile(lengths, 95)
percentile_99  = np.percentile(lengths, 99)
percentile_100 = np.percentile(lengths, 100)

plt.axvline(percentile_90,  color='orange', linestyle='--', linewidth=2, label=f'90th Percentile: {percentile_90:.0f}')
plt.axvline(percentile_95,  color='red',    linestyle='--', linewidth=2, label=f'95th Percentile: {percentile_95:.0f}')
plt.axvline(percentile_99,  color='green',  linestyle='--', linewidth=2, label=f'99th Percentile: {percentile_99:.0f}')
plt.axvline(percentile_100, color='purple', linestyle='--', linewidth=2, label=f'100th Percentile: {percentile_100:.0f}')

plt.title(f"MultiOFF {safe_model_name} Token Length Distribution on Training Data", fontsize=13, fontweight='bold', pad=20)
plt.xlabel("Number of Tokens", fontsize=12)
plt.ylabel("Frequency (Count)", fontsize=12)
plt.ylim(0, 100)
plt.legend(title="Statistics", loc='upper left', bbox_to_anchor=(1, 1), frameon=True)

sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Sets max_token_length based on the 99th percentile of token lengths.
# Caps sequences to cover 99% of samples without over-padding.

max_token_length = 512 # Default

if percentile_99 > 256:
    max_token_length = 512
elif percentile_99 > 128:
    max_token_length = 256
elif percentile_99 > 64:
    max_token_length = 128
elif percentile_99 > 32:
    max_token_length = 64
elif percentile_99 > 16:
    max_token_length = 32

print(max_token_length)

In [ ]:
def tokenize_function(batch):
    """Tokenizes a batch of text samples with padding and truncation."""
    return tokenizer(batch["sentence"], padding=True, truncation=True, max_length=max_token_length)

In [ ]:
# Image features were extracted offline using EfficientNetV2L at 380x380.
# Loaded from a compressed .npz archive to avoid re-running the image encoder.

IMG_NAME = "EfficientNetV2L"

feature_file = "multi3hate_features.npz"
image_features = np.load(feature_file)["feature"]

In [ ]:
# Assigns a fold index (0-4) to each sample using StratifiedKFold,
# preserving label distribution across folds. Fold IDs are stored directly
# in the dataframe so filtering by fold is straightforward during cross-validation.

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

folds = np.zeros(len(df_final), dtype=int)
for fold, (_, val_idx) in enumerate(skf.split(df_final, df_final["label"])):
    folds[val_idx] = fold

df_final["fold"] = folds

In [ ]:
# Attaches pre-extracted image feature vectors, converts to a HuggingFace Dataset,
# tokenizes text, and sets the PyTorch format so the data collator receives tensors.

df_final['image_features'] = [image_features[i] for i in df_final.index]

all_dataset   = hf_datasets.Dataset.from_pandas(df_final)
tokenized_data = all_dataset.map(tokenize_function, batched=True)

# Guard against Dataset.map dropping the image_features column on some versions.
if 'image_features' not in tokenized_data.column_names:
    tokenized_data = tokenized_data.add_column("image_features", df_final['image_features'].tolist())

tokenized_data.set_format(type='torch', columns=['input_ids', 'attention_mask', 'image_features', 'label'])

In [ ]:
# These parameters are used directly for all 30 runs without Optuna search.
# Since the dataset is only consisting of 300 samples.
# Intended for stability evaluation using a known-good configuration.

PREDEFINED_PARAMS = {
    'learning_rate':       1e-5,
    'batch_size':          2,
    'num_train_epochs':    10,
    'weight_decay':        1e-3,
    'attention_dropout':   0.1,
    'activation_function': 'gelu',
    'num_attention_heads': 8,
    'warmup_ratio':        0.1,
    'lr_decay_type':       'linear',
}

all_run_results = []

In [ ]:
# Repeats 5-fold cross-validation 30 times to estimate metric variance under
# fixed hyperparameters. Each run uses the same SEED for fold assignment but
# benefits from natural training stochasticity (e.g., weight initialization,
# dropout, batch ordering).
#
# Per fold: trains a fresh MultimodalClassifier and evaluates on the held-out fold.
# Per run: averages metrics across all 5 folds.
# All run averages are collected in all_run_results.

for run_id in range(1, 31):
    run_fold_results = []

    for fold_id in range(5):
        print(f"\n{'='*20} RUN {run_id} | FOLD {fold_id} {'='*20}")

        train_fold = tokenized_data.filter(lambda x: x['fold'] != fold_id, keep_in_memory=True)
        val_fold   = tokenized_data.filter(lambda x: x['fold'] == fold_id,  keep_in_memory=True)

        config = AutoConfig.from_pretrained(model_name)
        config.attention_probs_dropout_prob = PREDEFINED_PARAMS['attention_dropout']
        config.hidden_act                   = PREDEFINED_PARAMS['activation_function']
        config.num_attention_heads          = PREDEFINED_PARAMS['num_attention_heads']

        model = MultimodalClassifier(
            num_labels=2,
            image_feature_size=1280,
            model_name=model_name,
            class_weights=train_class_weights
        )
        model.model.config = config
        model.to(device)

        training_args = TrainingArguments(
            output_dir="./temp_results_copy",
            learning_rate=PREDEFINED_PARAMS['learning_rate'],
            per_device_train_batch_size=PREDEFINED_PARAMS['batch_size'],
            per_device_eval_batch_size=PREDEFINED_PARAMS['batch_size'],
            lr_scheduler_type=PREDEFINED_PARAMS['lr_decay_type'],
            num_train_epochs=PREDEFINED_PARAMS['num_train_epochs'],
            weight_decay=PREDEFINED_PARAMS['weight_decay'],
            warmup_ratio=PREDEFINED_PARAMS['warmup_ratio'],
            eval_strategy="epoch",
            save_strategy="no",           # No checkpointing during cross-validation
            load_best_model_at_end=False,
            save_total_limit=1,
            metric_for_best_model="eval_loss",
            greater_is_better=False,
            remove_unused_columns=False,
            label_names=["labels"],
            seed=SEED,
            report_to="none"
        )

        trainer = MultimodalTrainer(
            model=model,
            args=training_args,
            train_dataset=train_fold,
            eval_dataset=val_fold,
            data_collator=MultimodalDataCollator(tokenizer=tokenizer),
            compute_metrics=compute_metrics
        )

        trainer.train()
        eval_metrics = trainer.evaluate()

        run_fold_results.append({
            "loss":      eval_metrics["eval_loss"],
            "accuracy":  eval_metrics["eval_accuracy"],
            "precision": eval_metrics["eval_precision"],
            "recall":    eval_metrics["eval_recall"],
            "f1":        eval_metrics["eval_f1"],
            "auc_pr":    eval_metrics["eval_auc_pr"],
        })

        del model, trainer
        gc.collect()
        torch.cuda.empty_cache()

    # Average all metrics across the 5 folds for this run
    run_avg = {
        key: np.mean([fold[key] for fold in run_fold_results])
        for key in run_fold_results[0]
    }
    run_avg["run"] = run_id
    all_run_results.append(run_avg)

In [ ]:
print(f"{Style.BRIGHT}{Fore.BLUE}{safe_model_name} FINAL RESULTS FROM 30 RUNS{Style.RESET_ALL}")

final_results = {}
for metric in all_run_results[0].keys():
    if "run" in metric.lower():
        continue
    values = [run[metric] for run in all_run_results]
    final_results[metric] = {
        "mean": np.mean(values),
        "std":  np.std(values)
    }

for metric, stats in final_results.items():
    print(f"{metric.upper():<10} Mean: {stats['mean']:.4f} | Std: {stats['std']:.4f}")

In [ ]:
# Output file setup with date and model name for traceability.

today = date.today().strftime("%Y-%m-%d")
safe_img_name = re.sub(r'[\\/*?:"<>|/]', '_', IMG_NAME)
output_dir = "Results CSV/V30 Trials"
os.makedirs(output_dir, exist_ok=True)
output_file = f"{output_dir}/{safe_model_name}_30_trials.csv"

In [ ]:
# Converts run results to a DataFrame, appends mean and std summary rows,
# attaches metadata columns, and saves to CSV.

results_df = pd.DataFrame(all_run_results)
results_df = results_df.round(4)

mean_row = round(results_df.mean(), 4)
std_row  = round(results_df.std(),  4)
mean_row.name = "MEAN"
std_row.name  = "STD"

results_df["date"]      = today
results_df["model"]     = safe_model_name
results_df["image_set"] = safe_img_name

results_df = pd.concat([results_df, mean_row.to_frame().T, std_row.to_frame().T])
results_df.to_csv(output_file, index=True)

print(f"\nResults saved to: {output_file}")

In [ ]:
# watch -n 0.1 nvidia-smi